# Extracción inicial de la red vial de Valencia (para limpieza posterior)


In [ ]:
import osmnx as ox
import matplotlib.pyplot as plt
import pandas as pd
import missingno as msno
import geopandas as gpd
import folium
import requests
import json
import time
import numpy as np
from tqdm import tqdm
import seaborn as sns


place = "València, Spain"

# Descargar el grafo de la red vial para coches
G = ox.graph_from_place(place, network_type="drive", simplify=False)

edges = ox.graph_to_gdfs(G, nodes=False, edges=True)

print(f"Tramos descargados: {len(edges):,}")
print(f"Columnas disponibles: {len(edges.columns)}")
print("Listado de columnas:", sorted(edges.columns.tolist()))

In [ ]:
edges.head(5)

In [ ]:
df = edges.reset_index(drop=True)


In [ ]:
df.head()

# Atributos de calles OSMnx / OpenStreetMap

| Campo | Descripción |
|--------|--------------|
| **access** | Regla legal de acceso a la vía. |
| **bridge** | Indica si el tramo va sobre un puente (`bridge=yes`). |
| **geometry** | Geometría del tramo como `LineString` o `MultiLineString` (coordenadas de la calle). |
| **highway** | Clase de carretera en OSM. |
| **junction** | Tipo de intersección especial si aplica, p. ej. `junction=roundabout` (rotonda) o `circular`. Útil para detectar glorietas. |
| **lanes** | Número total de carriles para vehículos de dos huellas (sin contar bici/moto si están segregados). |
| **length** | Longitud del tramo **en metros**, calculada según la geometría. |
| **maxspeed** | Límite de velocidad. |
| **name** | Nombre de la vía (calle o avenida). |
| **oneway** | Indica si la vía es de **sentido único** (`True` / `False`). En OSM puede existir `oneway=-1` para sentido inverso. |
| **osmid** | Identificador(es) del/los **way(s)** de OpenStreetMap del que proviene el edge; puede ser una lista si el tramo agrupa varios. |
| **ref** | Referencia de la carretera (por ejemplo, `A-7`, `CV-35`). Típico de carreteras numeradas. |
| **reversed** | `True` si OSMnx invirtió el orden de nodos respecto al way original. Útil al analizar dirección o sentido. |
| **tunnel** | Indica si el tramo discurre por un túnel (`tunnel=yes`). |
| **width** | Ancho de la calzada (en **metros**). |


# Análisis exploratorio de datos (EDA)

Para empezar queremos asegurarnos de que no haya duplicados, o en caso de haberlos que sean correctos

In [ ]:
df['osmid'].value_counts()[df['osmid'].value_counts() > 1]

Se observan bastantes duplicados, analicemos su origen

In [ ]:
osmid_ejemplo = 37290056 
df[df['osmid'] == osmid_ejemplo]

In [ ]:
print(df[df['osmid'] == osmid_ejemplo]["length"].sum()/1000)

Cada fila de nuestro dataset representa un tramo de calle dirigido, es decir, un segmento de la red vial entre dos nodos. No una carretera completa

Deberíamos analizar los duplicados creando atributos derivados de geometry (sacando punto A y B)

In [ ]:
df['punto_A'] = df.geometry.apply(lambda x: x.coords[0] if x else None)
df['punto_B'] = df.geometry.apply(lambda x: x.coords[-1] if x else None)

df[['lon_A', 'lat_A']] = pd.DataFrame(df['punto_A'].tolist(), index=df.index)
df[['lon_B', 'lat_B']] = pd.DataFrame(df['punto_B'].tolist(), index=df.index)


In [ ]:
df.head(5)

Veamos si hay duplicados en los tramos

In [ ]:
df.duplicated(subset=['punto_A', 'punto_B']).any()

Creemos el id por tramo

In [ ]:
df['id_tramo'] = range(1, len(df) + 1)

Perfecto! Sigamos con el análisis

Observemos la distribución por tipos de vía.

In [ ]:
df['highway'].value_counts(dropna=False)

In [ ]:
df['highway'].value_counts().plot(
    kind='bar',
    figsize=(10,5),
    title='Distribución de tipos de vía (highway)'
)
plt.xlabel('Tipo de vía')
plt.ylabel('Frecuencia')
plt.show()


Procedemos ahora con el estudio de la variable lanes

In [ ]:
df['lanes'] = pd.to_numeric(df['lanes'], errors='coerce')

df['lanes'].value_counts(dropna=False).sort_index()


In [ ]:
df['lanes'].value_counts(dropna=False).sort_index().plot(
    kind='bar',
    figsize=(8,5),
    title='Distribución de número de carriles (lanes)',
    color='skyblue',
    edgecolor='black'
)
plt.xlabel('Número de carriles')
plt.ylabel('Frecuencia')
plt.show()


Podemos observar que tiene un alto porcentaje de nulos

In [ ]:
print(round(df['lanes'].isna().mean() * 100), '% de nulos')

Analicemos si tiene alguna relación los valores faltantes con otras variables

In [ ]:
# Muestra la matriz de nulos (como en MICE)
msno.matrix(df)
plt.show()

A priori no parece que tenga una coocurrencia de nulos con otras variables, pero indaguemos más y veamos si se relaciona con variables categóricas clave.

In [ ]:
df['lanes_missing'] = df['lanes'].isna()

In [ ]:
(
    df.groupby('oneway')['lanes_missing']
    .mean()
    .sort_values()
    .plot(kind='barh', figsize=(8,6), color='skyblue')
)
plt.title('Porcentaje de tramos con Lanes nulo por sentido único')
plt.xlabel('Proporción de NaN en lanes')
plt.ylabel('Tipo de vía')
plt.show()

In [ ]:
(
    df.groupby('highway')['lanes_missing']
    .mean()
    .sort_values()
    .plot(kind='barh', figsize=(8,6), color='skyblue')
)
plt.title('Porcentaje de tramos con Lanes nulo por tipo de vía')
plt.xlabel('Proporción de NaN en lanes')
plt.ylabel('Tipo de vía')
plt.show()

Veamos si hay algún patrón en el mapa

In [ ]:
# Crear mapa centrado aproximadamente en Valencia
m = folium.Map(location=[39.47, -0.38], zoom_start=13, tiles='cartodb positron')

# Iterar sobre el DataFrame y dibujar cada tramo
for _, row in df.iterrows():
    color = 'red' if pd.isna(row['lanes']) else 'green'
    
    # Coordenadas: punto A → punto B
    coords = [(row['lat_A'], row['lon_A']), (row['lat_B'], row['lon_B'])]
    
    folium.PolyLine(
        locations=coords,
        color=color,
        weight=2.5,
        opacity=0.8
    ).add_to(m)


m

Tras el análisis de la variable lanes, se confirma que los valores nulos no presentan un patrón sistemático ni espacial, por lo que su ausencia parece deberse principalmente a lagunas en la información original de OpenStreetMap.

En consecuencia, se aplicará una estrategia de imputación mixta:

Imputación contextual: los valores nulos se rellenarán utilizando la mediana del número de carriles dentro de cada categoría de vía (highway), garantizando coherencia con la distribución real de los tipos de carretera.

Variable indicadora: se conservará una variable binaria (lanes_missing) que identifique si el valor original era nulo o no. Esto permite al modelo de aprendizaje automático captar la ausencia de información como una posible fuente de variabilidad, evitando introducir sesgo al forzar un valor artificialmente.

Desde el punto de vista teórico, mantener esta variable indicadora resulta relevante porque la ausencia de datos puede ser informativa por sí misma (por ejemplo, las vías menos documentadas suelen ser de menor jerarquía o tráfico). De este modo, el modelo podrá distinguir entre tramos con información completa y aquellos donde la carencia de datos podría reflejar características estructurales del entorno vial.

In [ ]:
df['lanes'] = df.groupby('highway')['lanes'].transform(
    lambda x: x.fillna(x.median())
)


In [ ]:
print(round(df['lanes'].isna().mean() * 100), '% de nulos')

Ahora analicemos max_speed

In [ ]:
print(round(df['maxspeed'].isna().mean() * 100), '% de nulos')

In [ ]:
df['maxspeed_missing'] = df['maxspeed'].isna().astype(int)

(
    df.groupby('highway')['maxspeed_missing']
    .mean()
    .sort_values()
    .plot(kind='barh', figsize=(8,6), color='#C82909')
)
plt.title('Porcentaje de tramos con maxspeed nulo por tipo de vía')
plt.xlabel('Proporción de NaN en maxspeed')
plt.ylabel('Tipo de vía')
plt.show()


In [ ]:
# Crear mapa centrado aproximadamente en Valencia
m = folium.Map(location=[39.47, -0.38], zoom_start=13, tiles='cartodb positron')

# Iterar sobre el DataFrame y dibujar cada tramo
for _, row in df.iterrows():
    color = 'red' if pd.isna(row['maxspeed']) else 'green'
    
    # Coordenadas: punto A → punto B
    coords = [(row['lat_A'], row['lon_A']), (row['lat_B'], row['lon_B'])]
    
    folium.PolyLine(
        locations=coords,
        color=color,
        weight=2.5,
        opacity=0.8
    ).add_to(m)


m


Analicemos los nulos en name

In [ ]:
print(round(df['name'].isna().mean() * 100), '% de nulos')

A pesar de tener un 19% de faltantes, no vamos a imputar esta variable debido a que no es una variable predictora

Veamos los nulos en width

In [ ]:
print(round(df['width'].isna().mean() * 100), '% de nulos')

Demasiados nulos, la eliminamos.

Se elimina asimismo la variable ref, ya que, además de presentar un elevado número de valores nulos, no aporta información relevante para los objetivos del análisis.

In [ ]:
df = df.drop(columns=["width", "ref"])

Estudiemos el comportamiento de las variables tunnel, acces, bridge y juction

In [ ]:
df[['tunnel', 'access', 'bridge', 'junction']].info()
print(df[['tunnel', 'access', 'bridge', 'junction']].isna().mean() * 100)

In [ ]:
for col in ['tunnel', 'access', 'bridge', 'junction']:
    print(f"\nDistribución de {col}:")
    print(df[col].value_counts(dropna=False))


Dado que las variables tunnel, bridge, junction y access presentan una gran cantidad de valores nulos y categorías poco frecuentes, se optó por binarizarlas con el fin de conservar únicamente la información estructural más relevante. De este modo, se crean indicadores simples que reflejan la presencia de túneles, puentes, rotondas y restricciones de acceso.

In [ ]:
# Binarización de variables estructurales
df['es_tunel'] = df['tunnel'].apply(lambda x: 1 if x == 'yes' else 0)
df['es_puente'] = df['bridge'].apply(lambda x: 1 if x == 'yes' else 0)
df['es_rotonda'] = df['junction'].apply(lambda x: 1 if x in ['roundabout', 'circular'] else 0)

# Variable de acceso restringido
df['acceso_restringido'] = df['access'].apply(
    lambda x: 1 if x in ['no', 'private', 'destination', 'permit'] else 0
)

# Verificación rápida
df[['es_tunel', 'es_puente', 'es_rotonda', 'acceso_restringido']].sum()


## Imputación maxspeed

Esta implementación aplica un enfoque de imputación basada en similitud espacial y semántica para estimar los valores faltantes de maxspeed en una red vial. En primer lugar, se calculan las coordenadas del punto medio de cada tramo para representar su posición geográfica. Luego, se construye un espacio de características que combina variables categóricas (como el tipo de vía), numéricas (como número de carriles, longitud o presencia de túneles y puentes) y espaciales (latitud y longitud). Estas últimas se ponderan mediante un parámetro spatial_weight que controla cuánto influye la cercanía geográfica en la similitud. Con esta matriz de rasgos se entrena un modelo de k-vecinos más cercanos (kNN), que busca para cada tramo sin velocidad conocida los tramos más parecidos entre los que sí la tienen y calcula una media ponderada por distancia. Finalmente, los valores improbables o faltantes se reemplazan mediante la mediana del límite de velocidad correspondiente al tipo de vía (highway), garantizando coherencia normativa. Este procedimiento permite una imputación contextual y consistente, respetando tanto la estructura espacial como las reglas viales implícitas del conjunto de datos.

In [ ]:
def _center_coords(row):
    # coordenadas del punto medio del tramo A–B
    lat_c = (row['lat_A'] + row['lat_B']) / 2
    lon_c = (row['lon_A'] + row['lon_B']) / 2
    return pd.Series({'lat_c': lat_c, 'lon_c': lon_c})

def impute_maxspeed_by_similarity(
    df,
    k=7,
    spatial_weight=0.5,   # cuánto pesa la distancia espacial en la similitud (0–1)
    scale_numeric=True
):
    df = df.copy()

    # 1) Asegurar features auxiliares
    if 'lat_A' not in df or 'lon_A' not in df or 'lat_B' not in df or 'lon_B' not in df:
        raise ValueError("Faltan columnas lat_A/lon_A/lat_B/lon_B")

    centers = df.apply(_center_coords, axis=1)
    df[['lat_c', 'lon_c']] = centers

    # Variables binarias (si no existen, crearlas a 0)
    for col in ['es_tunel', 'es_puente', 'es_rotonda', 'acceso_restringido']:
        if col not in df:
            df[col] = 0

    # Asegurar lanes numérico (si falta, intentar convertir)
    if df['lanes'].dtype == object:
        df['lanes'] = pd.to_numeric(df['lanes'], errors='coerce')

    # 2) Definir conjuntos
    known = df['maxspeed'].notna()
    unknown = ~known

    if known.sum() == 0:
        raise ValueError("No hay tramos con maxspeed conocido para entrenar la similitud.")

    # 3) Selección de rasgos
    cat_cols = ['highway']  # puedes añadir 'surface' si la tienes
    num_cols = [
        'lanes', 'length',
        'es_tunel', 'es_puente', 'es_rotonda', 'acceso_restringido'
    ]

    # Rasgos espaciales (con peso aparte)
    spatial_cols = ['lat_c', 'lon_c']

    # 4) Preprocesamiento (one-hot + escala opcional)
    preproc = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
            ('num', StandardScaler(with_mean=True, with_std=True) if scale_numeric else 'passthrough', num_cols),
        ],
        remainder='drop'
    )

    # Ajustar preproc con datos conocidos
    X_known_core = preproc.fit_transform(df.loc[known, cat_cols + num_cols])
    X_unknown_core = preproc.transform(df.loc[unknown, cat_cols + num_cols])

    # 5) Añadir componente espacial con su propio peso
    # Estandarizamos a var≈1 para que spatial_weight tenga sentido
    spatial_scaler = StandardScaler()
    S_known = spatial_scaler.fit_transform(df.loc[known, spatial_cols])
    S_unknown = spatial_scaler.transform(df.loc[unknown, spatial_cols])

    X_known = np.hstack([X_known_core, spatial_weight * S_known])
    X_unknown = np.hstack([X_unknown_core, spatial_weight * S_unknown])

    # 6) Vecinos más cercanos sobre tramos conocidos
    nbrs = NearestNeighbors(n_neighbors=min(k, X_known.shape[0]), algorithm='auto', metric='minkowski')
    nbrs.fit(X_known)

    distances, indices = nbrs.kneighbors(X_unknown)
    # Evitar división por cero: sumamos un épsilon
    eps = 1e-6
    weights = 1 / (distances + eps)
    weights = weights / weights.sum(axis=1, keepdims=True)

    y_known = pd.to_numeric(df.loc[known, 'maxspeed'], errors='coerce').values
    y_known[np.isnan(y_known)] = np.nanmedian(y_known)

    # 7) Predicción ponderada por distancia (kNN “a mano”)
    y_pred = []
    for w, idxs in zip(weights, indices):
        vals = y_known[idxs]
        # si hay NaN residuales, reemplazar por la mediana de los vecinos
        if np.isnan(vals).any():
            med = np.nanmedian(vals)
            vals = np.where(np.isnan(vals), med, vals)
        y_pred.append(np.sum(w * vals))
    y_pred = np.array(y_pred)

    # 8) Fallback: donde la predicción sea NaN o absurda, usar mediana por highway
    df_result = df.copy()
    df_result.loc[unknown, 'maxspeed_imputada'] = y_pred
    # Convertir maxspeed a numérico antes de calcular la mediana
    df['maxspeed_num'] = pd.to_numeric(df['maxspeed'], errors='coerce')


    # Fallback por mediana de highway
    mediana_por_highway = (
        df.loc[known]
          .groupby('highway')['maxspeed']
          .median()
    )

    def fallback(row):
        if pd.isna(row['maxspeed_imputada']) or row['maxspeed_imputada'] <= 0:
            return mediana_por_highway.get(row['highway'], np.nanmedian(y_known))
        return row['maxspeed_imputada']

    df_result.loc[unknown, 'maxspeed_imputada'] = df_result.loc[unknown].apply(fallback, axis=1)

    # 9) Integración final: mantener original y crear columna final
    df_result['maxspeed_final'] = df_result['maxspeed']
    df_result.loc[unknown, 'maxspeed_final'] = df_result.loc[unknown, 'maxspeed_imputada']

    # Limpieza de tipos
    df_result['maxspeed_final'] = pd.to_numeric(df_result['maxspeed_final'], errors='coerce')

    return df_result

# --- USO ---
df['maxspeed'] = pd.to_numeric(df['maxspeed'], errors='coerce')

df_out = impute_maxspeed_by_similarity(df, k=7, spatial_weight=0.6)
df_out[['maxspeed', 'maxspeed_imputada', 'maxspeed_final']].head(100)


In [ ]:
df['maxspeed_final'] = df_out['maxspeed_final']
df.columns

In [ ]:
# Filtramos valores no nulos y plausibles (0 < velocidad < 150 km/h)
df_plot = df[['maxspeed', 'maxspeed_final']].copy()
df_plot = df_plot.apply(pd.to_numeric, errors='coerce')
df_plot = df_plot[(df_plot['maxspeed_final'] > 0) & (df_plot['maxspeed_final'] < 150)]

plt.figure(figsize=(8,5))
sns.boxplot(data=df_plot[['maxspeed', 'maxspeed_final']])
plt.title('Comparación de distribuciones: original vs imputada')
plt.ylabel('Velocidad (km/h)')
plt.grid(alpha=0.3)
plt.show()



In [ ]:
plt.figure(figsize=(10,6))
sns.kdeplot(df_plot['maxspeed'], label='Original', fill=True, alpha=0.3, color='red')
sns.kdeplot(df_plot['maxspeed_final'], label='Imputada', fill=True, alpha=0.3, color='green')
plt.title('Comparación de densidades: maxspeed vs maxspeed_final')
plt.xlabel('Velocidad (km/h)')
plt.ylabel('Densidad')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


Podemos observar claramente que a pesar de haber imputado una variable que originalmente tenía un 78% de valores faltantes, se ha podido respetar muy bien la distribución de valores

# Guardamos los datos

In [ ]:
nuevo_orden = ['osmid',  'id_tramo', 'name', 'highway', 'lanes', 'lanes_missing', 'maxspeed',  'maxspeed_missing', 'maxspeed_final', 'oneway', 'reversed',
    'length', 'junction', 'tunnel', 'access', 'bridge', 'geometry',
    'punto_A', 'punto_B', 'lon_A', 'lat_A', 'lon_B', 'lat_B', 'es_tunel', 'es_puente',
    'es_rotonda', 'acceso_restringido']


df = df[nuevo_orden]


In [ ]:
df.to_csv('../data/datos_valencia_limpios.csv', index=False, encoding='utf-8')
